# My Big Board
This is a new Big Board that combines data from different sources to provide a clearer view of each prospect.

## 1. Load Players

Load the ordered list of players manually built with my own preferences:

In [1]:
import sys
sys.path.insert(0, '../../src')

import pandas as pd
import data_loader

RAW_FILE = "arif-consensous-bb.csv"

df = pd.read_csv(data_loader.DATA_RAW_DIR / RAW_FILE)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns")
# df.head()

Loaded 300 rows and 4 columns


## 2. Add Data from Brugler The Beast

In [2]:

# Reload Brugler data
data_loader.save_brugler_data()

df_brugler = pd.read_csv(data_loader.DATA_PROCESSED_DIR / data_loader.BRUGLER_FILE)
df_brugler.head()

COLUMN_NAME = "Brugler"
df = df.merge(
    df_brugler[["NAME", "YEAR","HEIGHT","WEIGHT","40-YD (10-YD)","HAND SIZE","ARM LENGTH","WING SPAN","AGE","GRADE"]].rename(columns={"GRADE": COLUMN_NAME}),
    left_on="Player",
    right_on="NAME",
    how="left",
).drop(columns=["NAME"])

#df.head()

## 3. Add MockDraftDatabase.com Rank

In [3]:
df_mddb = pd.read_csv(data_loader.DATA_RAW_DIR / "nflmockdraftdatabase-2026-04-15.csv", dtype={"RANK": "Int64"})
COLUMN_NAME = "MockDB"

df = df.merge(
    df_mddb[["NAME", "RANK"]].rename(columns={"RANK": COLUMN_NAME}),
    left_on="Player",
    right_on="NAME",
    how="left",
).drop(columns=["NAME"])

# Keep rank as nullable integer after the left merge.
df[COLUMN_NAME] = df[COLUMN_NAME].astype("Int64")

#df.head()

## 4. Add PFF Grades

In [4]:
df_pff = pd.read_csv(data_loader.DATA_RAW_DIR / "pff-big-board-2026-04-13.csv", dtype={"Rank": "Int64"})
COLUMN_NAME = "PFF-Rank"

df = df.merge(
    df_pff[["Player", "Rank", "PFF Grade"]].rename(columns={"Rank": COLUMN_NAME}),
    left_on="Player",
    right_on="Player",
    how="left",
)

# Keep rank as nullable integer after the left merge.
df[COLUMN_NAME] = df[COLUMN_NAME].astype("Int64")


## 5. Add RAS Grades

In [5]:
df_ras = pd.read_csv(data_loader.DATA_RAW_DIR / "ras-2026-04-14.csv")
COLUMN_NAME = "RAS"

df = df.merge(
    df_ras[["Name", "RAS"]],
    left_on="Player",
    right_on="Name",
    how="left",
).drop(columns=["Name"])

# Show a dash when RAS is missing or blank.
df[COLUMN_NAME] = df[COLUMN_NAME].replace("", pd.NA).fillna("-")

## 6. Add NFL grades

In [6]:
df_nfl = pd.read_csv(data_loader.DATA_RAW_DIR / "nfl-grades.csv")
COLUMN_NAME = "NFL-Rank"

df = df.merge(
    df_nfl[["NAME", "GRADE", "NEXTGEN"]].rename(columns={"GRADE": COLUMN_NAME, "NEXTGEN": "NextGen"}),
    left_on="Player",
    right_on="NAME",
    how="left",
).drop(columns=["NAME"])

### 7. Add Vikings data

Add comments related to Vikings: Top30 visits, Private interviews or work-outs, interviews during the Scouting Combine or ProDay.

In [7]:
df_nfl = pd.read_csv(data_loader.DATA_RAW_DIR / "vikings-predraft-visits.csv")
COLUMN_NAME = "Vikings Comments"

df = df.merge(
    df_nfl[["NAME", "COMMENTS"]].rename(columns={"COMMENTS": COLUMN_NAME}),
    left_on="Player",
    right_on="NAME",
    how="left",
).drop(columns=["NAME"])

## 8. Save Big Board
Generate an output CSV with all the data.

In [8]:
data_loader.save_processed(df, "my-big-board.csv")
print("Saved to data/processed/my-big-board.csv")

Saved to data/processed/my-big-board.csv
